# Multilingual Text-JEPA Representation Geometry — Colab Runner
Thin 4-cell wrapper around `src/` and `scripts/`. Cached layer tensors stay on fast local Colab SSD (`data/cache/`), while final tables and 300-DPI figures are saved directly to Google Drive.

In [ ]:
# Cell 1: Mount Google Drive, clone/sync repository, and install dependencies
import getpass
import os
from google.colab import drive

drive.mount('/content/drive')

GITHUB_USER = 'alisakour'
REPO_NAME = 'multilingual-text-jepa-geometry'
REPO_DIR = f'/content/{REPO_NAME}'
DRIVE_OUT = '/content/drive/MyDrive/jepa_platonic_benchmark_v2'
os.makedirs(DRIVE_OUT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    token = getpass.getpass('GitHub PAT (required if private repo; leave blank if public): ').strip()
    if token:
        repo_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        repo_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    !git clone {repo_url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Cell 2: Build the strictly quad-parallel mLAMA benchmark (AR, EN, ZH, FR)
!python -m src.data.build_benchmark --raw_dir data/raw --out_dir data/processed

In [ ]:
# Cell 3: Extract & cache all 13 layers ([CLS] + MeanPool) on GPU (~3-4 min total)
!python scripts/run_extraction.py --config configs/models.yaml --cache_dir data/cache

In [ ]:
# Cell 4: Compute Tables 1-5 (BH-FDR) and 300-DPI Figures 1-2 (PNG + PDF) to Google Drive
!python scripts/run_analysis.py --cache_dir data/cache --out_dir "{DRIVE_OUT}"

# Optional CPU-only MeanPool ablation across all cached models (takes ~2-3 min, zero GPU needed):
# !python scripts/run_analysis.py --cache_dir data/cache --pooling_override mean --out_dir "{DRIVE_OUT}/meanpool_control" --no_plots